# step8: Val 段调执行参数 

---
## 1 Setup & Load Locked Artifacts

In [122]:
import json
import numpy as np
import pandas as pd
import joblib
import pickle
from pathlib import Path
from scipy.stats import spearmanr

DATA = Path('data')

with open(DATA / 'splits.json') as f:
    splits = json.load(f)
val_start = pd.Timestamp(splits['val']['start'])
val_end   = pd.Timestamp(splits['val']['end'])

with open(DATA / 'signal_bins.json') as f:
    signal_bins = json.load(f)

lookup_table   = pd.read_parquet(DATA / 'lookup_table.parquet')
regime_forward = pd.read_parquet(DATA / 'regime_forward.parquet')
regime_post    = pd.read_parquet(DATA / 'regime_posterior.parquet')
X_P_residual   = pd.read_parquet(DATA / 'X_P_residual.parquet')
regime_all     = regime_forward["regime"]
ohlcv          = pd.read_parquet(DATA / 'spy_minute_clean.parquet')

# ── 加载 regime-conditional 模型 ──────────────────────────
with open(DATA / 'P_model.pkl', 'rb') as f:
    P_bundle = pickle.load(f)

regime_models   = P_bundle["regime_models"]   # {0: Ridge, 1: Ridge}
scaler_resid    = P_bundle["scaler_resid"]
feature_names   = P_bundle["feature_names"]
alpha_locked    = P_bundle["alpha_locked"]
REGIME_LABELS   = P_bundle["regime_labels"]
K_FINAL         = P_bundle["K_FINAL"]
feature_set     = P_bundle["feature_set"]

print('val window :', val_start.date(), '->', val_end.date())
print('lookup_table shape   :', lookup_table.shape)
print('signal_bins keys     :', list(signal_bins.keys()))
print('regime_forward range :', regime_forward.index.min().date(), '->', regime_forward.index.max().date())
print('residual columns     :', list(X_P_residual.columns))
print('K_FINAL              :', K_FINAL)
print('feature_names        :', feature_names)
print('alpha_locked         :', alpha_locked)
print('feature_set          :', feature_set)

val window : 2023-04-26 -> 2024-08-27
lookup_table shape   : (10, 11)
signal_bins keys     : ['bin_quantiles', 'bin_edges', 'note']
regime_forward range : 2015-06-25 -> 2025-12-31
residual columns     : ['CloseLoc', 'SignedVolume_z252', 'PriceVolCorr_z252', 'Overnight', 'Mom_60d', 'Mom_20d']
K_FINAL              : 2
feature_names        : ['CloseLoc_resid', 'SignedVolume_z252_resid', 'PriceVolCorr_z252_resid', 'Overnight_resid', 'Mom_60d_resid', 'Mom_20d_resid']
alpha_locked         : {0: 1.4384, 1: 0.0695}
feature_set          : residual_only_4dim


---
## 2  Inspect Lookup Table

In [123]:
print('=== lookup_table full view ===')
print(lookup_table.to_string())
print()
print('confidence value counts:')
print(lookup_table['confidence'].value_counts())

=== lookup_table full view ===
   regime  signal_bin    n  mean_ret  median_ret   std_ret  win_rate  sharpe_ann  ci95_low  ci95_high confidence
0       0           0  148 -0.004162   -0.001257  0.015273  0.425676   -4.325515 -0.006947  -0.001767       high
1       0           1  147 -0.000163    0.000885  0.012327  0.510204   -0.209582 -0.002213   0.001260        low
2       0           2  147  0.000111    0.000329  0.014457  0.523810    0.121729 -0.001684   0.001999        low
3       0           3  147  0.000414    0.001629  0.013053  0.544218    0.503873 -0.002145   0.002827        low
4       0           4  148  0.005228    0.004994  0.018610  0.601351    4.459779  0.003205   0.007161       high
5       1           0  165 -0.000833   -0.000282  0.008529  0.484848   -1.551281 -0.002457   0.000432        low
6       1           1  164  0.000437    0.000454  0.005075  0.567073    1.367666 -0.000264   0.001050        low
7       1           2  164  0.001107    0.000818  0.005592  0.597

---
## 3 Build Val Daily Frame (Regime, Posterior, Signal, r_next)

In [124]:
# ── 列名映射 ──────────────────────────────────────────────
print('feature_names from P_bundle :', feature_names)
print('X_P_residual columns        :', list(X_P_residual.columns))

col_map = {}
for fn in feature_names:
    base = fn.replace('_resid', '')
    if fn in X_P_residual.columns:
        col_map[fn] = fn
    elif base in X_P_residual.columns:
        col_map[fn] = base
    else:
        raise KeyError(f'cannot locate {fn} (or {base}) in X_P_residual')


# ── 构造目标变量 r_next ───────────────────────────────────
daily_close = ohlcv.groupby(ohlcv.index.normalize())['close'].last()
daily_close.index = pd.to_datetime(daily_close.index).tz_localize(None)
log_ret = np.log(daily_close / daily_close.shift(1))
r_next  = log_ret.shift(-1).rename('r_next')

# ── 切出 val 段并去掉边界 ─────────────────────────────────
val_dates         = X_P_residual.loc[val_start:val_end].index
resid_val         = X_P_residual.loc[val_dates]
resid_val_trimmed = resid_val.iloc[5:]

# ── 按 regime 分组预测 ────────────────────────────────────
ordered_cols = [col_map[fn] for fn in feature_names]
X_raw    = resid_val_trimmed[ordered_cols].values
X_scaled = scaler_resid.transform(X_raw)

seg_regime = regime_all.reindex(resid_val_trimmed.index)

pred = np.full(len(resid_val_trimmed), np.nan)
for r in range(K_FINAL):
    mask = (seg_regime.values == r)
    if mask.sum() == 0:
        continue
    pred[mask] = regime_models[r].predict(X_scaled[mask])

signal = pd.Series(pred, index=resid_val_trimmed.index, name='signal')

# ── 组装 val_frame ────────────────────────────────────────
reg_col  = regime_forward.columns[0]
post_arr = regime_post.loc[resid_val_trimmed.index].values
post_max = pd.Series(
    post_arr.max(axis=1),
    index=resid_val_trimmed.index,
    name='posterior_max'
)

val_frame = pd.concat([
    regime_forward.loc[resid_val_trimmed.index, reg_col].rename('regime'),
    post_max,
    signal,
    r_next.loc[resid_val_trimmed.index].rename('r_next'),
], axis=1)
val_frame['r_next'] = r_next.reindex(val_frame.index).values

print()
print('val_frame shape :', val_frame.shape)
print('regime col used :', reg_col)
print('NaN per column  :')
print(val_frame.isna().sum())
print()
print(val_frame.head(8))
print(val_frame.tail(8))

feature_names from P_bundle : ['CloseLoc_resid', 'SignedVolume_z252_resid', 'PriceVolCorr_z252_resid', 'Overnight_resid', 'Mom_60d_resid', 'Mom_20d_resid']
X_P_residual columns        : ['CloseLoc', 'SignedVolume_z252', 'PriceVolCorr_z252', 'Overnight', 'Mom_60d', 'Mom_20d']

val_frame shape : (331, 4)
regime col used : regime
NaN per column  :
regime           0
posterior_max    0
signal           0
r_next           0
dtype: int64

            regime  posterior_max    signal    r_next
date                                                 
2023-05-03       1       0.925002  0.001186 -0.006936
2023-05-04       0       0.956181  0.002847  0.018196
2023-05-05       1       0.511509  0.000193  0.000436
2023-05-08       1       0.993813  0.001105 -0.004443
2023-05-09       1       0.999617  0.002225  0.004467
2023-05-10       1       0.991567  0.000438 -0.001552
2023-05-11       1       0.997134  0.002007 -0.001335
2023-05-12       1       0.999735  0.001321  0.003396
            regime  pos

---
## 4 Map Signal to Bin & Cell

In [125]:
edges_by_regime = {}
for k, v in signal_bins['bin_edges'].items():
    edges_by_regime[int(k)] = np.array(
        [-np.inf if e == '-inf' else (np.inf if e == 'inf' else float(e)) for e in v]
    )

print('reconstructed edges by regime:')
for k, v in edges_by_regime.items():
    print(f'  regime {k}: {v}')

def assign_bin(signal_value, regime, edges_dict):
    edges = edges_dict[int(regime)]
    edges_inner = edges[1:-1]
    return int(np.digitize([signal_value], edges_inner)[0])

val_frame['signal_bin'] = val_frame.apply(
    lambda r: assign_bin(r['signal'], r['regime'], edges_by_regime), axis=1
)

lk = lookup_table.reset_index() if lookup_table.index.name is not None else lookup_table.copy()
print('lookup_table columns after reset:', list(lk.columns))

saved_index=val_frame.index
val_frame = val_frame.merge(
    lk[['regime', 'signal_bin', 'mean_ret', 'confidence']],
    on=['regime', 'signal_bin'],
    how='left',
)
val_frame.index=saved_index

print('val_frame shape after merge :', val_frame.shape)
print('rows with missing cell merge:', val_frame['confidence'].isna().sum())
print()
print('regime x signal_bin counts in val:')
print(val_frame.groupby(['regime', 'signal_bin']).size().unstack(fill_value=0))
print()
print('confidence distribution in val:')
print(val_frame['confidence'].value_counts(dropna=False))

reconstructed edges by regime:
  regime 0: [       -inf -0.00240544 -0.00058123  0.00098111  0.00306459         inf]
  regime 1: [       -inf -0.00012934  0.0003079   0.0006911   0.00112378         inf]
lookup_table columns after reset: ['regime', 'signal_bin', 'n', 'mean_ret', 'median_ret', 'std_ret', 'win_rate', 'sharpe_ann', 'ci95_low', 'ci95_high', 'confidence']
val_frame shape after merge : (331, 7)
rows with missing cell merge: 0

regime x signal_bin counts in val:
signal_bin   0   1   2   3    4
regime                         
0            2  14  10  20    8
1           60  34  32  45  106

confidence distribution in val:
confidence
low     289
high     42
Name: count, dtype: int64


---
## 5 Posterior Distribution 

In [126]:
print('=== posterior_max quantiles by regime (val) ===')
print(val_frame.groupby('regime')['posterior_max'].describe(percentiles=[.1, .25, .5, .75, .9]))
print()



=== posterior_max quantiles by regime (val) ===
        count      mean       std       min       10%       25%       50%  \
regime                                                                      
0        54.0  0.935270  0.113036  0.545162  0.767950  0.906101  0.998840   
1       277.0  0.965967  0.092519  0.511509  0.902015  0.993531  0.999633   

             75%       90%       max  
regime                                
0       1.000000  1.000000  1.000000  
1       0.999879  0.999932  0.999962  



---
## 6 Decision Function & Grid Evaluator

In [127]:
def daily_position(row, unc_thresh, low_conf_size):
    if pd.isna(row['confidence']):
        return 0.0
    if row['posterior_max'] < unc_thresh:
        return 0.0
    if row['confidence'] == 'no_trade':
        return 0.0
    direction = np.sign(row['mean_ret'])
    if row['confidence'] == 'high':
        return float(direction)
    # low confidence：直接用 low_conf_size 控制仓位
    # low_conf_size=0 等价于原来的 no_trade
    return low_conf_size * float(direction)

def evaluate_combo(frame, unc_thresh, low_conf_size, cost_bp=2.0):
    df = frame.copy()
    df['position'] = df.apply(
        lambda r: daily_position(r, unc_thresh, low_conf_size), axis=1
    )
    df['turnover']  = df['position'].diff().abs().fillna(df['position'].abs())
    df['gross_pnl'] = df['position'] * df['r_next']
    df['cost']      = df['turnover'] * (cost_bp / 1e4)
    df['net_pnl']   = df['gross_pnl'] - df['cost']

    valid   = df.dropna(subset=['r_next'])
    nonzero = valid[valid['position'] != 0]
    if len(nonzero) >= 5:
        ic, _ = spearmanr(nonzero['position'], nonzero['r_next'])
    else:
        ic = np.nan

    n_total      = len(valid)
    n_zero       = (valid['position'] == 0).sum()
    pct_no_trade = n_zero / n_total if n_total > 0 else np.nan

    by_regime = valid.groupby('regime').agg(
        n=('position', 'size'),
        n_traded=('position', lambda s: (s != 0).sum()),
        net_pnl=('net_pnl', 'sum'),
        hit_rate=('gross_pnl', lambda s: (s[s != 0] > 0).mean() if (s != 0).any() else np.nan),
    )

    summary = {
        'unc_thresh':    unc_thresh,
        'low_conf_size': low_conf_size,
        'rank_IC':       ic,
        'net_pnl_total': df['net_pnl'].sum(),
        'pct_no_trade':  pct_no_trade,
        'mean_turnover': df['turnover'].mean(),
        'n_traded':      (valid['position'] != 0).sum(),
    }
    return summary, df, by_regime

---
## 7 Run 12-Combo Grid

In [128]:
grid = []
combo_details = {}

for ut in [0.5, 0.6, 0.7]:
    for lcs in [0.0, 0.25, 0.5, 0.75]:
        summary, df_combo, by_reg = evaluate_combo(val_frame, ut, low_conf_size=lcs)
        grid.append(summary)
        key = (ut, lcs)
        combo_details[key] = {'frame': df_combo, 'by_regime': by_reg}

grid_df = pd.DataFrame(grid)
grid_df = grid_df.sort_values('rank_IC', ascending=False).reset_index(drop=True)

print(f'=== {len(grid_df)}-combo grid (sorted by rank_IC desc) ===')
print(grid_df.to_string(index=False))

=== 12-combo grid (sorted by rank_IC desc) ===
 unc_thresh  low_conf_size   rank_IC  net_pnl_total  pct_no_trade  mean_turnover  n_traded
        0.6           0.50 -0.072335       0.020649      0.021148       0.324773       324
        0.6           0.75 -0.072335       0.016100      0.021148       0.381420       324
        0.6           0.25 -0.072335       0.025198      0.021148       0.268127       324
        0.5           0.25 -0.073096       0.024315      0.000000       0.259063       331
        0.5           0.50 -0.073096       0.018884      0.000000       0.306647       331
        0.5           0.75 -0.073096       0.013453      0.000000       0.354230       331
        0.5           0.00 -0.092240       0.029747      0.873112       0.211480        42
        0.6           0.00 -0.092240       0.029747      0.873112       0.211480        42
        0.7           0.25 -0.098553      -0.006958      0.054381       0.260574       313
        0.7           0.50 -0.098553      -

In [129]:
print('=== val activation per (regime, signal_bin) ===')
activation = val_frame.groupby(['regime', 'signal_bin']).agg(
    n_val=('signal', 'size'),
    mean_signal_val=('signal', 'mean'),
    mean_r_next=('r_next', 'mean'),
).reset_index()

activation = activation.merge(
    lk[['regime', 'signal_bin', 'n', 'mean_ret', 'confidence']].rename(
        columns={'n': 'n_train', 'mean_ret': 'mean_ret_train'}
    ),
    on=['regime', 'signal_bin'], how='right',
).fillna({'n_val': 0})

print(activation.to_string(index=False))
print()
print('cells with n_val == 0:')
print(activation[activation['n_val'] == 0])

=== val activation per (regime, signal_bin) ===
 regime  signal_bin  n_val  mean_signal_val  mean_r_next  n_train  mean_ret_train confidence
      0           0      2        -0.003451     0.002732      148       -0.004162       high
      0           1     14        -0.001324     0.002298      147       -0.000163        low
      0           2     10         0.000288     0.006400      147        0.000111        low
      0           3     20         0.001967    -0.000508      147        0.000414        low
      0           4      8         0.004007     0.006658      148        0.005228       high
      1           0     60        -0.000728     0.001698      165       -0.000833        low
      1           1     34         0.000120     0.003311      164        0.000437        low
      1           2     32         0.000487    -0.000127      164        0.001107       high
      1           3     45         0.000922    -0.000360      164        0.000564        low
      1           4   

---
## 12  Lock Final Combo & Persist Outputs

In [130]:
final_params = {
    'low_conf_size': 0.5,
    'regime_uncertainty_threshold': 0.6,
    'cost_per_turnover_bp': 2.0,
    'min_samples_per_cell': 30,
    'r0_bins': 5,
    'r1_bins': 5,
}

final_summary, final_frame, final_by_regime = evaluate_combo(
    val_frame,
    final_params['regime_uncertainty_threshold'],
    final_params['low_conf_size'],
    cost_bp=final_params['cost_per_turnover_bp'],
)

print('=== final combo summary ===')
for k, v in final_summary.items():
    print(f'  {k:20s}: {v}')
print()
print('=== final per-regime breakdown ===')
print(final_by_regime)
print()
print('=== full grid (for record) ===')
print(grid_df.to_string(index=False))

=== final combo summary ===
  unc_thresh          : 0.6
  low_conf_size       : 0.5
  rank_IC             : -0.07233484096840426
  net_pnl_total       : 0.020648617782004303
  pct_no_trade        : 0.021148036253776436
  mean_turnover       : 0.324773413897281
  n_traded            : 324

=== final per-regime breakdown ===
          n  n_traded   net_pnl  hit_rate
regime                                   
0        54        53  0.054897  0.528302
1       277       271 -0.034249  0.479705

=== full grid (for record) ===
 unc_thresh  low_conf_size   rank_IC  net_pnl_total  pct_no_trade  mean_turnover  n_traded
        0.6           0.50 -0.072335       0.020649      0.021148       0.324773       324
        0.6           0.75 -0.072335       0.016100      0.021148       0.381420       324
        0.6           0.25 -0.072335       0.025198      0.021148       0.268127       324
        0.5           0.25 -0.073096       0.024315      0.000000       0.259063       331
        0.5         

---
## 13 Build & Save val_backtest

In [131]:
val_backtest = final_frame[[
    'regime', 'posterior_max', 'signal', 'signal_bin',
    'mean_ret', 'confidence', 'r_next',
    'position', 'turnover', 'gross_pnl', 'cost', 'net_pnl',
]].copy()

val_backtest['cum_net_pnl'] = val_backtest['net_pnl'].fillna(0).cumsum()

val_backtest.to_parquet(DATA / 'val_backtest.parquet')

print('val_backtest saved:', DATA / 'val_backtest.parquet')
print('shape   :', val_backtest.shape)
print('columns :', list(val_backtest.columns))
print()
print('=== val_backtest head ===')
print(val_backtest.head(8))
print()
print('=== val_backtest tail ===')
print(val_backtest.tail(8))
print()
print('=== val PnL trajectory milestones ===')
n = len(val_backtest)
for q in [0.25, 0.5, 0.75, 1.0]:
    idx = int(n * q) - 1
    row = val_backtest.iloc[idx]
    label = val_backtest.index[idx]
    if hasattr(label, 'date'):
        label_str = str(label.date())
    else:
        label_str = f'row {label}'
    print(f'  {q*100:>5.1f}% ({label_str}): cum_net_pnl = {row["cum_net_pnl"]:.4f}')

val_backtest saved: data\val_backtest.parquet
shape   : (331, 13)
columns : ['regime', 'posterior_max', 'signal', 'signal_bin', 'mean_ret', 'confidence', 'r_next', 'position', 'turnover', 'gross_pnl', 'cost', 'net_pnl', 'cum_net_pnl']

=== val_backtest head ===
            regime  posterior_max    signal  signal_bin  mean_ret confidence  \
date                                                                           
2023-05-03       1       0.925002  0.001186           4  0.001115        low   
2023-05-04       0       0.956181  0.002847           3  0.000414        low   
2023-05-05       1       0.511509  0.000193           1  0.000437        low   
2023-05-08       1       0.993813  0.001105           3  0.000564        low   
2023-05-09       1       0.999617  0.002225           4  0.001115        low   
2023-05-10       1       0.991567  0.000438           2  0.001107       high   
2023-05-11       1       0.997134  0.002007           4  0.001115        low   
2023-05-12       1

---
## 14 Build & Save execution_params.json

In [132]:
grid_records = grid_df.to_dict(orient='records')
for r in grid_records:
    for k, v in r.items():
        if isinstance(v, (np.integer,)):
            r[k] = int(v)
        elif isinstance(v, (np.floating,)):
            r[k] = None if np.isnan(v) else float(v)
        elif isinstance(v, (np.bool_,)):
            r[k] = bool(v)

per_regime_clean = {
    str(int(reg)): {
        'n': int(row['n']),
        'n_traded': int(row['n_traded']),
        'net_pnl': float(row['net_pnl']),
        'hit_rate': None if pd.isna(row['hit_rate']) else float(row['hit_rate']),
    }
    for reg, row in final_by_regime.iterrows()
}

execution_params = {
    'final_params': final_params,
    'final_val_summary': {
        'rank_IC': None if pd.isna(final_summary['rank_IC']) else float(final_summary['rank_IC']),
        'net_pnl_total': float(final_summary['net_pnl_total']),
        'pct_no_trade': float(final_summary['pct_no_trade']),
        'mean_turnover': float(final_summary['mean_turnover']),
        'n_traded': int(final_summary['n_traded']),
        'n_total_val_days': int(len(val_frame)),
    },
    'final_per_regime': per_regime_clean,
    'grid_full': grid_records,
    'flags': {
        'pct_no_trade_below_expected_band': bool(final_summary['pct_no_trade'] < 0.30),
        'r2_direction_drift_in_val': True,
        'r0_val_n': 14,
        'r0_val_n_below_threshold': True,
    },
    'notes': {
        'low_conf_action_rationale': (
            'half_size_directional chosen to preserve test-set statistical power. '
            'Trade-off: bypasses Step 7 confidence semantics by using mean_ret sign as a weak directional signal. '
            'Step 9 attribution must check this assumption first if r1/r2 fail in test.'
        ),
        'r2_drift_note': (
            'Cell 10 shows r2_bin=2 train -0.149% vs val +0.568% (sign-flipped) and '
            'r2_bin=4 train -0.0008% vs val -0.177%. Train direction structure may not hold on test.'
        ),
        'r0_locked_from_train': (
            'r0 has only 14 samples in val; bin edges, mean_ret, confidence labels all '
            'inherited from Step 7 train statistics without modification.'
        ),
        'unc_thresh_economic_meaning': (
            'unc=0.6 mainly gates r2 low-posterior transition days, consistent with '
            'r2 being the "elevated normal transition" regime per Step 5a.'
        ),
    },
}

with open(DATA / 'execution_params.json', 'w') as f:
    json.dump(execution_params, f, indent=2)

print('execution_params saved:', DATA / 'execution_params.json')
print()
print(json.dumps(execution_params, indent=2))

execution_params saved: data\execution_params.json

{
  "final_params": {
    "low_conf_size": 0.5,
    "regime_uncertainty_threshold": 0.6,
    "cost_per_turnover_bp": 2.0,
    "min_samples_per_cell": 30,
    "r0_bins": 5,
    "r1_bins": 5
  },
  "final_val_summary": {
    "rank_IC": -0.07233484096840426,
    "net_pnl_total": 0.020648617782004303,
    "pct_no_trade": 0.021148036253776436,
    "mean_turnover": 0.324773413897281,
    "n_traded": 324,
    "n_total_val_days": 331
  },
  "final_per_regime": {
    "0": {
      "n": 54,
      "n_traded": 53,
      "net_pnl": 0.05489728949366836,
      "hit_rate": 0.5283018867924528
    },
    "1": {
      "n": 277,
      "n_traded": 271,
      "net_pnl": -0.03424867171166407,
      "hit_rate": 0.4797047970479705
    }
  },
  "grid_full": [
    {
      "unc_thresh": 0.6,
      "low_conf_size": 0.5,
      "rank_IC": -0.07233484096840426,
      "net_pnl_total": 0.020648617782004303,
      "pct_no_trade": 0.021148036253776436,
      "mean_turnov